In [1]:
# ------------------------------------------------------------------------------
# 🛡️ AUTOMATED BENCHMARK PIPELINE: MODEL REPLACEMENT (SCALING ATTACK)
# ------------------------------------------------------------------------------
# Cenário: O atacante escala seus pesos para substituir o modelo global.
# Fator de Escala (Gamma): 100x
# Objetivo: Testar a robustez contra anomalias de MAGNITUDE.
# ------------------------------------------------------------------------------

import subprocess
import sys
import time
import os
import glob
import traceback
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Instalação Silenciosa
try:
    import hdbscan
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "hdbscan"])
    import hdbscan

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
INPUT_PATTERN = "/content/drive/MyDrive/Colab Notebooks/CSAI4CPS/Datasets/*consolidado.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Benchmark_ModelReplacement_Results" # Pasta Nova

# Parâmetros do Ataque DE ESCALA
SCALING_FACTOR = 100.0  # Gamma = 100 (O vetor malicioso é 100x maior)

# Parâmetros Estatísticos
N_ROUNDS_STATISTICAL = 5
INITIAL_SEED = 42
NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES = [f"S{i}" for i in range(17)]

# Defesa
ALPHA_PENALTY = 0.85
BETA_RECOVERY = 0.10
TRUST_THRESHOLD = 0.2
ABSOLUTE_MAJORITY_RULE = True
FLAME_NOISE_SIGMA = 0.001

# Modelo
HIDDEN_LAYERS = (64,)
LOCAL_EPOCHS = 3

# ==============================================================================
# 2. FUNÇÕES CORE
# ==============================================================================
def create_model(seed):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu',
                         solver='adam', random_state=seed, max_iter=1)

def get_weights(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training(global_weights, X, y, classes, ref_model, seed):
    model = create_model(seed)
    if len(X) >= 5: model.partial_fit(X[:5], y[:5], classes=classes)
    else: model.partial_fit(X, y, classes=classes)
    set_weights(model, global_weights.copy(), ref_model)
    rng = np.random.RandomState(seed)
    for _ in range(LOCAL_EPOCHS):
        p = rng.permutation(len(X))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights(model) - global_weights

# ==============================================================================
# 3. LÓGICA DE EXPERIMENTO (MODEL REPLACEMENT)
# ==============================================================================
def run_single_seed_scaling(df, target_col, seed):
    y_raw = df[target_col].values.astype(int)
    X_raw = df.drop(columns=[target_col]).values
    classes = np.unique(y_raw)

    X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=seed)
    X_root, X_eval, y_root, y_eval = train_test_split(X_test, y_test, test_size=0.5, stratify=y_test, random_state=seed)

    scaler = StandardScaler().fit(X_train)
    X_root_sc = scaler.transform(X_root)
    X_eval_sc = scaler.transform(X_eval)

    X_backdoor = X_eval_sc.copy()
    if X_backdoor.shape[1] > 0:
        X_backdoor[:, -1] = 99.0

    strategies = ["FedAvg", "Krum", "FLAME", "FLTrust", "Adaptive_Ultimate"]
    results = []

    for strategy in strategies:
        g_model = create_model(seed)
        g_model.partial_fit(X_eval_sc[:10], y_eval[:10], classes=classes)
        g_weights = get_weights(g_model)
        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES):
            start_t = time.time()

            shards_X = np.array_split(X_train, len(NODES))
            shards_y = np.array_split(y_train, len(NODES))
            local_updates = []

            # Progressão de Atacantes (Igual ao primeiro teste)
            atk_cnt = 0
            if stage_idx > 0: atk_cnt = min((stage_idx - 1) // 4 + 1, len(NODES))
            atk_idx = list(range(atk_cnt))

            for i, node in enumerate(NODES):
                X_n = shards_X[i].copy()
                y_n = shards_y[i].copy()

                is_attacker = i in atk_idx

                # Injeção de Backdoor
                if is_attacker:
                    n_p = int(len(y_n) * 0.5)
                    if n_p > 0:
                        rng = np.random.RandomState(seed + stage_idx + i)
                        p_idx = rng.choice(len(y_n), n_p, replace=False)
                        if X_n.shape[1] > 0: X_n[p_idx, -1] = 99.0
                        y_n[p_idx] = 1

                delta = local_training(g_weights, scaler.transform(X_n), y_n, classes, g_model, seed)

                # --- APLICAÇÃO DO MODEL REPLACEMENT (ESCALA) ---
                if is_attacker:
                    delta = delta * SCALING_FACTOR  # Aumenta magnitude em 100x

                local_updates.append(delta)

            # Agregação
            root_delta = local_training(g_weights, X_root_sc, y_root, classes, g_model, seed)
            root_norm = np.linalg.norm(root_delta) + 1e-12
            agg_update = np.zeros_like(g_weights)

            if strategy == "FedAvg":
                # FedAvg vai explodir aqui porque aceita a magnitude bruta
                agg_update = np.mean(local_updates, axis=0)

            elif strategy == "Krum":
                # Krum seleciona um vetor. Se selecionar o atacante (por estar perto geometricamente), explode.
                k = max(1, len(local_updates) - 3)
                dists = cdist(np.stack(local_updates), np.stack(local_updates))
                scores = [np.sum(np.sort(d)[1:k+1]) for d in dists]
                agg_update = local_updates[np.argmin(scores)]

            elif strategy == "FLAME":
                # FLAME tem clipping, deve resistir melhor
                stk = np.stack(local_updates)
                dists = cosine_distances(stk)
                labels = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=2, allow_single_cluster=True).fit_predict(dists)
                u, c = np.unique(labels, return_counts=True)
                valid = u[u != -1]
                sel = stk[np.where(labels == valid[np.argmax(c[u != -1])])[0]] if len(valid) > 0 else stk

                # Clipping Mecanismo
                med = np.median(np.linalg.norm(sel, axis=1))
                clip = [v * min(1, med/(np.linalg.norm(v)+1e-12)) for v in sel]
                agg_update = np.mean(clip, axis=0) + np.random.normal(0, FLAME_NOISE_SIGMA * med, g_weights.shape)

            elif strategy == "FLTrust":
                # FLTrust normaliza explicitamente. Deve sobreviver.
                scores = []
                for d in local_updates:
                    sim = np.dot(root_delta, d) / (root_norm * (np.linalg.norm(d) + 1e-12))
                    scores.append(max(0, sim))
                    # Normalização Crucial:
                    norm_up = d * (root_norm / (np.linalg.norm(d) + 1e-12))
                    agg_update += max(0, sim) * norm_up
                if sum(scores) > 0: agg_update /= sum(scores)

            elif strategy == "Adaptive_Ultimate":
                valid_cnt = 0
                maj_thr = (len(NODES) // 2) + 1
                trust_w = []
                for i, d in enumerate(local_updates):
                    sim = np.dot(root_delta, d) / (root_norm * (np.linalg.norm(d) + 1e-12))
                    old = trust_memory[NODES[i]]
                    if sim < 0: new = 0.0
                    elif (sim - old) < 0: new = old + (ALPHA_PENALTY * (sim - old))
                    else: new = old + (BETA_RECOVERY * (sim - old))
                    new = np.clip(new, 0.0, 1.0)
                    trust_memory[NODES[i]] = new
                    if new < TRUST_THRESHOLD: w = 0.0
                    else: w, valid_cnt = new, valid_cnt + 1
                    trust_w.append(w)

                if valid_cnt < maj_thr:
                    agg_update = np.zeros_like(g_weights)
                else:
                    tot = 0
                    for i, w in enumerate(trust_w):
                        if w > 0:
                            # Normalização Crucial (Aqui que vencemos o Scaling Attack)
                            norm_up = local_updates[i] * (root_norm / (np.linalg.norm(local_updates[i]) + 1e-12))
                            agg_update += w * norm_up
                            tot += w
                    if tot > 0: agg_update /= tot

            g_weights += agg_update
            set_weights(g_model, g_weights, g_model)

            results.append({
                "Seed": seed,
                "Strategy": strategy,
                "Stage": stage_idx,
                "F1": f1_score(y_eval, g_model.predict(X_eval_sc), average='macro'),
                "ASR": np.mean(g_model.predict(X_backdoor) == 1),
                "Time": time.time() - start_t
            })

    return results

# ==============================================================================
# 4. ORQUESTRADOR
# ==============================================================================
def process_all_datasets_scaling():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')

    files = glob.glob(INPUT_PATTERN)
    if not files:
        files = [f for f in os.listdir('.') if f.endswith('consolidado.csv')]

    files = [f for f in files if "RESULTADOS" not in f and "Benchmark" not in f and "Summary" not in f]

    if not files:
        print("❌ Nenhum dataset válido encontrado.")
        return

    print(f"📦 Encontrados {len(files)} datasets. Iniciando MODEL REPLACEMENT (Gamma=100)...")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    global_summary_list = []

    for idx, fpath in enumerate(files):
        fname = os.path.basename(fpath)
        dataset_name = fname.replace(".csv", "").replace("_consolidado", "")
        print(f"\n▶️ [{idx+1}/{len(files)}] Processando: {dataset_name}")

        try:
            df = pd.read_csv(fpath)
            cols_ignore = ['Dataset_Source', 'device_category', 'host', 'ts', 'date', 'time']
            df = df.drop(columns=[c for c in cols_ignore if c in df.columns], errors='ignore')

            target_col = None
            for c in df.columns:
                if c.lower() in ['class', 'target', 'label', 'attack', 'benign']:
                    target_col = c; break
            if target_col is None: target_col = df.columns[-1]

            if df[target_col].dtype == object:
                le = LabelEncoder()
                df[target_col] = le.fit_transform(df[target_col].astype(str))

            y = df[target_col]
            X = df.drop(columns=[target_col]).select_dtypes(include=[np.number])
            df_clean = X.copy()
            df_clean['TARGET_FINAL'] = y

            dataset_results = []
            for i in range(N_ROUNDS_STATISTICAL):
                seed = INITIAL_SEED + i
                print(f"   ↳ Seed {seed}...", end="\r")
                res = run_single_seed_scaling(df_clean, 'TARGET_FINAL', seed)
                dataset_results.extend(res)

            print(f"   ✅ Concluído.")

            dataset_dir = os.path.join(OUTPUT_DIR, dataset_name)
            os.makedirs(dataset_dir, exist_ok=True)

            full_df = pd.DataFrame(dataset_results)
            full_df.to_csv(os.path.join(dataset_dir, "Raw_Results_Scaling.csv"), index=False)

            # Plot
            plt.figure(figsize=(10, 5))
            sns.lineplot(data=full_df, x='Stage', y='ASR', hue='Strategy', errorbar='sd')
            plt.title(f'ASR Model Replacement (x100) - {dataset_name}')
            plt.axvline(9, color='r', linestyle='--')
            plt.savefig(os.path.join(dataset_dir, "Plot_Scaling.png"))
            plt.close()

            last = full_df[full_df['Stage'] == 16].groupby('Strategy').mean(numeric_only=True).reset_index()
            last['Dataset'] = dataset_name
            global_summary_list.append(last)

        except Exception as e:
            print(f"   ❌ ERRO FATAL: {str(e)}")
            continue

    if global_summary_list:
        final = pd.concat(global_summary_list)
        final.to_csv(os.path.join(OUTPUT_DIR, "RELATORIO_SCALING_GERAL.csv"), index=False)
        print(f"\n🏆 FIM. Relatório Scaling salvo.")

if __name__ == "__main__":
    process_all_datasets_scaling()

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


📦 Encontrados 9 datasets. Iniciando MODEL REPLACEMENT (Gamma=100)...

▶️ [1/9] Processando: Philips_B120N10_Baby_Monitor_mirai_attacks
   ✅ Concluído.

▶️ [2/9] Processando: Philips_B120N10_Baby_Monitor_gafgyt_attacks
   ✅ Concluído.

▶️ [3/9] Processando: Danmini_Doorbell_mirai_attacks
   ✅ Concluído.

▶️ [4/9] Processando: Danmini_Doorbell_gafgyt_attacks
   ✅ Concluído.

▶️ [5/9] Processando: Ecobee_Thermostat_mirai_attacks
   ✅ Concluído.

▶️ [6/9] Processando: Ecobee_Thermostat_gafgyt_attacks
   ✅ Concluído.

▶️ [7/9] Processando: Provision_PT_838_Security_Camera_mirai_attacks
   ✅ Concluído.

▶️ [8/9] Processando: Samsung_SNH_1011_N_Webcam_gafgyt_attacks
   ✅ Concluído.

▶️ [9/9] Processando: Ennio_Doorbell_gafgyt_attacks
   ✅ Concluído.

🏆 FIM. Relatório Scaling salvo.
